In [1]:
from langchain_ollama import OllamaEmbeddings
# 创建向量模型,我们今天使用ollama

ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

# 初始化向量数据库客户端对象
from pymilvus import MilvusClient
from app.core.config import settings

# 初始化客户端
milvus_client = MilvusClient(uri=settings.rag.milvus_url)

# Collection名称:集合,指的就是表名字
collection_name = "my_collection_1"

In [2]:
# 定义稀疏向量匹配
def sparse_query(question: str) -> list[list['dict']]:
    """
    输入用户问题匹配向量
    :param question: 用户问题
    :return: 返回的结果是一个列表,返回多条向量匹配的结果,每一个子列表是匹配的文本片段
    """
    # 设置查询参数
    results = milvus_client.search(
        # 集合名
        collection_name=collection_name,
        # 根据稀疏向量匹配
        anns_field='sparse',
        # 输入用户问题的向量
        data=[question],
        # 最多匹配三条
        limit=3,
        # 返回三个字段的结果
        output_fields=['id','content']
    )
    return results

In [3]:
import json
# 调用函数遍历结果打印
results = sparse_query('教育的负面结果是什么')

# 循环遍历
for hits in results:
    for hit in  hits:
        print(json.dumps(hit,indent=2,ensure_ascii=False))

{
  "id": 15,
  "distance": 3.485706329345703,
  "entity": {
    "content": "# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。",
    "id": 15
  }
}
{
  "id": 9,
  "distance": 0.9465993046760559,
  "entity": {
    "content": "## 第二节 教育的定义  \n### （一）教育的定义  \n广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  \n狭义的教育就是指学校教育。  \n教育的要素：教育者、受教育者、教育影响（主要是教育内容）。",
    "id": 9
  }
}
{
  "id": 21,
  "distance": 0.8447741866111755,
  "entity": {
    "content": "### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。",
    "id": 21
  }
}


In [4]:
# 关键词匹配结果
results = sparse_query('教育漫画是谁写的')

# 循环遍历
for hits in results:
    for hit in  hits:
        print(json.dumps(hit,indent=2,ensure_ascii=False))

{
  "id": 7,
  "distance": 2.0690438747406006,
  "entity": {
    "content": "### （七）教育学创立时期代表人物及主要思想  \n**昆体良**：西方第一个专门论述教育问题的教育家。  \n**培根**：在《论科学的价值和发展》中首次把“教育学”作为一门独立的科学确立下来。  \n**夸美纽斯**（教育学之父、教育史上哥白尼）：代表作《大教学论》是教育学成为一门独立学科的标志。提出“泛智教育”、“班级授课制”和“学年制”；把教师比喻为太阳底下最光辉的事业。  \n**康德**：教育学作为一门课程在大学里讲授，始于康德。  \n**赫尔巴特**（传统教育代表、科学教育学之父）：《普通教育学》是教育学作为一门规范、独立的学科正式诞生的标志。提出以伦理学和心理学作为教育学的基础；教学要有教育性；三中心论：教师、教材、课堂；四阶段论：清楚、联想、系统和方法。  \n**杜威**（现代教育代表、实用主义哲学之父、儿童中心主义论、教育无目的论）：《民主主义与教育》提出教育的本质：教育即生活、教育即生长、教育即经验的改组或改造；学校即社会；从做中学；连带学习；三中心论：儿童、经验、活动；五步教学法：设疑—分析—假设—推断—验证。  \n**卢梭**：《爱弥儿》，提倡自然主义教育思想，认为教育的任务应该使儿童归于自然。  \n**洛克**：《教育漫画》，提出白板说，倡导绅士教育。  \n**裴斯泰洛奇**：《林哈德与葛笃德》，教育心理学化。  \n**斯宾塞**：反对思辨，主张用实证方法研究知识价值；生活预备说；科学知识最有价值，制定以科学知识为核心的课程体系。  \n**陶行知**：提出生活教育理论：生活即教育，社会即学校，教学做合一；“捧着一颗心来”（师德）；伟大的人民教育家。  \n**蔡元培**：思想自由，兼容并包；以美育代宗教；学界泰斗，人世楷模。",
    "id": 7
  }
}
{
  "id": 9,
  "distance": 0.9465993046760559,
  "entity": {
    "content": "## 第二节 教育的定义  \n### （一）教育的定义  \n广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，